# FER-2013: Exploratory Data Analysis

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

PROCESSED = Path("../data/processed")
RAW = Path("../data/raw/fer2013.csv")

EMOTIONS = ("angry", "disgust", "fear", "happy", "sad", "surprise", "neutral")

print("Processed dir exists:", PROCESSED.exists())
print("Raw CSV exists:", RAW.exists())
print("Processed files:", sorted(p.name for p in PROCESSED.glob("*.npy")))

## Counting dataset

In [ ]:
splits = {}
for split in ("train", "val", "test"):
    X = np.load(PROCESSED / f"X_{split}.npy")
    y = np.load(PROCESSED / f"y_{split}.npy")
    splits[split] = (X, y)
    print(f"{split:5s}  X={X.shape} {X.dtype}  y={y.shape} {y.dtype}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (split, (_, y)) in zip(axes, splits.items()):
    counts = np.bincount(y, minlength=7)
    bars = ax.bar(EMOTIONS, counts, color=sns.color_palette("Set2", 7))
    ax.set_title(f"{split}  (n={len(y)})")
    ax.set_ylabel("count" if split == "train" else "")
    ax.tick_params(axis="x", rotation=30)
    for bar, c in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            c,
            str(c),
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.suptitle("FER-2013 class distribution per split", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
_, y_train = splits["train"]
counts = np.bincount(y_train, minlength=7)
df = pd.DataFrame({"emotion": EMOTIONS, "count": counts})
df["pct"] = 100 * df["count"] / df["count"].sum()
df["ratio_vs_max"] = df["count"].max() / df["count"]
print(df.to_string(index=False))
print(f"\nMax/min ratio: {counts.max() / counts.min():.1f}x")

## Visualizing samples

In [ ]:
X_train, y_train = splits["train"]

n_per_class = 6
fig, axes = plt.subplots(7, n_per_class, figsize=(n_per_class * 1.6, 7 * 1.6))

for row, emotion_idx in enumerate(range(7)):
    idxs = np.where(y_train == emotion_idx)[0][:n_per_class]
    for col, i in enumerate(idxs):
        ax = axes[row, col]
        ax.imshow(X_train[i], cmap="gray", vmin=0, vmax=255)
        ax.axis("off")
        if col == 0:
            ax.set_title(EMOTIONS[emotion_idx], loc="left", fontsize=10)

plt.suptitle("FER-2013 samples", y=0.995)
plt.tight_layout()
plt.show()

## Some statistical features

In [ ]:
X_train_flat = X_train.reshape(len(X_train), -1).astype(np.float32)

print(f"global mean: {X_train_flat.mean() / 255:.4f}")
print(f"global std:  {X_train_flat.std() / 255:.4f}")
print(f"min: {X_train.min()}  max: {X_train.max()}")

# Per-channel-equivalent stats if we later replicate to 3 channels.
# For grayscale, mean and std are the same across channels.
mean = X_train_flat.mean() / 255
std = X_train_flat.std() / 255

# Per-emotion stats (mean and std of pixel intensity for each class)
per_class = []
for i, emo in enumerate(EMOTIONS):
    Xc = X_train[y_train == i].reshape(-1).astype(np.float32) / 255
    per_class.append({"emotion": emo, "mean": Xc.mean(), "std": Xc.std()})
print(pd.DataFrame(per_class).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

means_per_image = X_train.reshape(len(X_train), -1).mean(axis=1)
axes[0].hist(means_per_image, bins=50, color="steelblue")
axes[0].set_title("Mean pixel intensity per image")
axes[0].set_xlabel("mean intensity (0–255)")

stds_per_image = X_train.reshape(len(X_train), -1).std(axis=1)
axes[1].hist(stds_per_image, bins=50, color="indianred")
axes[1].set_title("Pixel std per image")
axes[1].set_xlabel("std (0–255)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

bins = np.arange(0, 257, 4)  # 64 bins of width 4

for ax, (split, (X, _)) in zip(axes, splits.items()):
    ax.hist(X.reshape(-1), bins=bins, color="steelblue", alpha=0.8)
    ax.set_title(f"{split}  (n={X.size:,} pixels)")
    ax.set_xlabel("pixel value (0–255)")
    ax.set_ylabel("count" if split == "train" else "")

plt.suptitle("Pixel intensity distribution per split", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
X_train, y_train = splits["train"]

fig, axes = plt.subplots(4, 2, figsize=(12, 12))
axes = axes.flat

for i, emo in enumerate(EMOTIONS):
    ax = axes[i]
    pixels = X_train[y_train == i].reshape(-1)
    ax.hist(pixels, bins=bins, density=True, color=sns.color_palette("Set2", 7)[i])
    ax.set_title(f"{emo}  (n={len(pixels):,} px)")
    ax.set_xlabel("pixel value")
    ax.set_ylabel("density")

axes[-1].axis("off")
plt.suptitle("Per-emotion pixel intensity distribution (train, normalized)", y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
X_train, _ = splits["train"]
flat = X_train.reshape(len(X_train), -1)

per_image_mean = flat.mean(axis=1)
per_image_std = flat.std(axis=1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(per_image_mean, bins=80, color="steelblue")
axes[0, 0].set_title("Per-image mean intensity")
axes[0, 0].set_xlabel("mean (0–255)")

axes[0, 1].hist(per_image_std, bins=80, color="indianred")
axes[0, 1].set_title("Per-image std")
axes[0, 1].set_xlabel("std (0–255)")

# 2D joint
axes[1, 0].hist2d(per_image_mean, per_image_std, bins=60, cmap="viridis")
axes[1, 0].set_xlabel("mean")
axes[1, 0].set_ylabel("std")

# Fraction of pixels equal to 0 or 255 per image (clipping indicator)
frac_clipped = ((flat == 0) | (flat == 255)).mean(axis=1)
axes[1, 1].hist(frac_clipped, bins=80, color="seagreen")
axes[1, 1].set_title("Fraction of pixels at 0 or 255")
axes[1, 1].set_xlabel("fraction clipped")

plt.tight_layout()
plt.show()

## Key findings and decisions

**Data.** Splits are representative (identical histogram shape). Pixel distribution is broad, right-skewed, peak ~150–180, with saturation spikes at 0/255 (~3–4%). Images are healthy: no outliers, no filtering needed. Severe class imbalance: `disgust` ≈ 547 vs `happy` ≈ 8,989. Brightness is a confound: `disgust` darker, `surprise` brighter.

**Decisions.**
- ResNet18, ImageNet-pretrained, full fine-tune (no frozen layers).
- 48×48 $\rightarrow$ 224×224, replicate 1$\rightarrow$3 channels.
- ImageNet normalization (pretrained BN expects it).
- Augmentation: `HorizontalFlip(0.5)`, `RandomAffine(10°, 0.05, 0.95–1.05)`, `ColorJitter(0.2, 0.2)`; skip CLAHE/blur/vertical flip.
- Imbalance: `WeightedRandomSampler` + class-weighted `CrossEntropyLoss`.
- Keep 7 classes; merge `disgust` only if F1 < 0.30.

**Target.** ~73-75% test accuracy. Honest SoTA ~82%; >90% numbers in the literature use non-official splits.

**Next.** `transforms.py` + `datamodule.py` with four regression tests: shape/dtype, val determinism, train stochasticity, dataloader batch contract.